In [ ]:
!pip install sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')

In [ ]:
sentences = [
    "I love playing football",
    "Soccer is my favourite sport",
    "I enjoy coding in Python",
    "Machine learning is fascinating",
    "Deep learning uses neural networks",
    "I hate rainy days",
    "The weather is terrible today",
    "Cats are better than dogs",
    "I prefer dogs over cats",
    "Python is great for data science"
]

In [ ]:
embeddings = model.encode(sentences)
print("Shape:", embeddings.shape)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
# Compare sentence 0 and sentence 1
sim = cosine_similarity([embeddings[0]], [embeddings[1]])
print(f"'{sentences[0]}' vs '{sentences[1]}'")
print(f"Similarity: {sim[0][0]:.4f}")

# Compare sentence 0 and sentence 5
sim2 = cosine_similarity([embeddings[3]], [embeddings[8]])
print(f"\n'{sentences[3]}' vs '{sentences[8]}'")
print(f"Similarity: {sim2[0][0]:.4f}")

In [ ]:
query = "I enjoy machine learning"
query_embedding = model.encode([query])

In [ ]:
# Calculate similarity with all sentences
similarities = cosine_similarity(query_embedding, embeddings)[0]

In [ ]:
ranked = sorted(zip(similarities, sentences), reverse=True)

In [ ]:
print(f"Query: '{query}'\n")
print("Most similar sentences:")
for score, sentence in ranked:
    print(f"{score:.4f} — {sentence}")

In [ ]:
from sklearn.decomposition import PCA

# Reduce 384 dimensions to 2 for visualization
pca = PCA(n_components=2)
reduced = pca.fit_transform(embeddings)
reduced

In [ ]:
# Plot
plt.figure(figsize=(10, 7))
for i, sentence in enumerate(sentences):
    plt.scatter(reduced[i, 0], reduced[i, 1])
    plt.annotate(sentence[:30], (reduced[i, 0], reduced[i, 1]),
                fontsize=8, ha='right')

plt.title("Sentence Embeddings Visualized in 2D")
plt.tight_layout()
plt.show()

## Exp-1

In [ ]:
my_sentences = [
     "Python is used for machine learning and data science",
     "JavaScript is mainly used for web development",
     "Neural networks are inspired by the human brain",
     "Machine learning is fascinating",
     "Deep learning uses neural networks"
]

In [ ]:
embeddings = model.encode(my_sentences)
print("Shape:", embeddings.shape)

In [ ]:
# Compare sentence 0 and sentence 1
sim = cosine_similarity([embeddings[2]], [embeddings[4]])
print(f"'{my_sentences[2]}' vs '{my_sentences[4]}'")
print(f"Similarity: {sim[0][0]:.4f}")

# Compare sentence 0 and sentence 5
sim2 = cosine_similarity([embeddings[0]], [embeddings[3]])
print(f"\n'{my_sentences[0]}' vs '{my_sentences[3]}'")
print(f"Similarity: {sim2[0][0]:.4f}")

In [ ]:
pca = PCA(n_components=2)
reduced = pca.fit_transform(embeddings)
print(reduced)

# Plot
plt.figure(figsize=(10, 7))
for i, sentence in enumerate(my_sentences):
    plt.scatter(reduced[i, 0], reduced[i, 1])
    plt.annotate(sentence[:30], (reduced[i, 0], reduced[i, 1]),
                fontsize=8, ha='right')

plt.title("Sentence Embeddings Visualized in 2D")
plt.tight_layout()
plt.show()

In [ ]:
# These share NO common words but mean the same thing
s1 = "The movie was fantastic"
s2 = "I really enjoyed the film"

e1 = model.encode([s1])
e2 = model.encode([s2])
sim = cosine_similarity(e1, e2)[0][0]
print(f"Similarity: {sim:.4f}")

## Exp-2: Multilingual embeddings

In [ ]:
model_multi = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

mixed_sentences = [
    "I love machine learning",      # English
    "मुझे मशीन लर्निंग पसंद है",      # Hindi — same meaning
    "J'adore l'apprentissage automatique",  # French — same meaning
    "I hate machine learning",      # English — opposite meaning
]

In [ ]:
mixed_embeddings = model_multi.encode(mixed_sentences)
sim_matrix = cosine_similarity(mixed_embeddings)

In [ ]:
print("Similarity Matrix:")
for i, s1 in enumerate(mixed_sentences):
    for j, s2 in enumerate(mixed_sentences):
        print(f"{sim_matrix[i][j]:.3f}", end=" ")
    print()

In [ ]:
# Same meaning, different languages
print("English vs Hindi:", sim_matrix[0][1])
print("English vs French:", sim_matrix[0][2])

# Same language, opposite meaning
print("English vs opposite:", sim_matrix[0][3])

## Exp-3: Simple semantic search

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')

In [ ]:
documents = [
    "Python is used for machine learning and data science",
    "JavaScript is mainly used for web development",
    "Neural networks are inspired by the human brain",
    "React is a popular frontend framework",
    "Transformers revolutionized natural language processing",
    "SQL is used to query relational databases",
    "Docker containers help deploy applications consistently",
    "Gradient descent optimizes neural network weights"
]

In [ ]:
doc_embeddings = model.encode(documents)

def semantic_search(query, top_k=3):
    query_embedding = model.encode([query])
    similarities = cosine_similarity(query_embedding, doc_embeddings)[0]
    ranked = sorted(zip(similarities, documents), reverse=True)

    print(f"Query: '{query}'\n")
    for score, doc in ranked[:top_k]:
        print(f"{score:.4f} — {doc}")
    print()

In [ ]:
semantic_search("how do machines learn")
semantic_search("frontend web tools")
semantic_search("database management")

In [ ]:
semantic_search("I want to build a website")
semantic_search("how to optimize a model")
semantic_search("storing and retrieving data")
semantic_search("what is the best pizza topping")

In [ ]:
def semantic_search(query, top_k=3, threshold=0.3):
    query_embedding = model.encode([query])
    similarities = cosine_similarity(query_embedding, doc_embeddings)[0]
    ranked = sorted(zip(similarities, documents), reverse=True)

    print(f"Query: '{query}'\n")
    for score, doc in ranked[:top_k]:
        if score >= threshold:
            print(f"{score:.4f} — {doc}")
        else:
            print(f"No relevant document found for this query")
            break
    print()

# Test with completely irrelevant query
semantic_search("what is the best pizza topping")

## Exp-4

In [ ]:
confusing_pairs = [
    ("I love cats", "I hate cats"),           # opposite meaning, similar words
    ("bank of a river", "bank for money"),     # same word, different meaning
    ("He is cold", "The weather is cold"),     # same word, different context
    ("Python is great", "Python bit me"),      # Python language vs Python snake
]

for s1, s2 in confusing_pairs:
    e1 = model.encode([s1])
    e2 = model.encode([s2])
    sim = cosine_similarity(e1, e2)[0][0]
    print(f"{sim:.4f} — '{s1}' vs '{s2}'")

## Exp-5

In [ ]:
import seaborn as sns

# Redefine all 10 sentences
sentences = [
    "I love playing football",
    "Soccer is my favourite sport",
    "I enjoy coding in Python",
    "Machine learning is fascinating",
    "Deep learning uses neural networks",
    "I hate rainy days",
    "The weather is terrible today",
    "Cats are better than dogs",
    "I prefer dogs over cats",
    "Python is great for data science"
]

# Re-embed all 10
embeddings = model.encode(sentences)
print("Shape:", embeddings.shape)  # Should be (10, 384)

# Now rerun heatmap
sim_matrix = cosine_similarity(embeddings)
short_labels = [s[:25] for s in sentences]

plt.figure(figsize=(9, 6))
sns.heatmap(sim_matrix,
            xticklabels=short_labels,
            yticklabels=short_labels,
            annot=True,
            fmt='.2f',
            cmap='YlOrRd')
plt.title("Sentence Similarity Heatmap")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()